In [3]:
# ============================================================
# 0. EINSTELLUNGEN
# ============================================================
BILD_PFAD   = None     # z.B. "katze.jpg". None = Auswahl aus einer Liste der Bilder im Ordner
ZIEL_NAME   = None     # z.B. "tabby". None = wahrscheinlichste Klasse
N           = 4        # Raster N x N (3 = 9 Felder, 4 = 16 Felder)
NUM_SAMPLES = 5        # Dropout-Durchläufe pro Bildvariante
BUDGET      = 1024     # Kombinationen bei der Näherung (nur bei mehr als 10 Feldern)

# ============================================================
# 1. Imports und Gerät (GPU, wenn vorhanden)
# ============================================================
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"      # nicht unterstützte Operationen laufen dann auf der CPU
                                                      # (muss VOR dem Import von torch stehen)
import glob, urllib.request
import numpy as np
import matplotlib.pyplot as plt
import torch
from PIL import Image
from torchvision.models import vgg16, VGG16_Weights
import shapiq
from shapiq.approximator import KernelSHAP
from probly.method import dropout
from probly.representer import representer
from probly.quantification import quantify

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print("Rechnet auf:", device)

# ============================================================
# 2. Bild auswählen
# ============================================================

import os
from PIL import Image

BILD_PFAD = "andrew-s-ouo1hbizWwo-unsplash-scaled.jpg"      # <-- hier den Pfad eintragen, z.B. "katze.jpg" oder "/Users/tristanfuchs/Bilder/hund.png"

BILD_PFAD = os.path.expanduser(BILD_PFAD)               # "~" wird zum Home-Ordner
if not os.path.exists(BILD_PFAD):
    raise FileNotFoundError(f"Bild nicht gefunden: {BILD_PFAD}\nAktueller Ordner: {os.getcwd()}")

bild = Image.open(BILD_PFAD).convert("RGB")
print("Verwendetes Bild:", BILD_PFAD, "  Größe:", bild.size)
plt.imshow(bild); plt.axis("off"); plt.show()

# ============================================================
# 3. VGG16 auf die GPU legen, Bild vorbereiten, Zielklasse bestimmen
# ============================================================
gewichte = VGG16_Weights.IMAGENET1K_V1
vgg = vgg16(weights=gewichte).eval().to(device)            # Netz auf die GPU
klassen = gewichte.meta["categories"]

x = gewichte.transforms()(bild).unsqueeze(0).to(device)    # Bild auf die GPU
with torch.no_grad():
    wahrsch = torch.softmax(vgg(x), dim=1)[0].cpu()        # fürs Anzeigen zurück auf die CPU

top5 = torch.topk(wahrsch, 5)
print("\nTop 5 von VGG16:")
for p, i in zip(top5.values, top5.indices):
    print(f"  {int(i):4d}  {klassen[int(i)]:28s} {p.item():.3f}")

if ZIEL_NAME is None:
    ziel = int(wahrsch.argmax())
else:
    treffer = [i for i, k in enumerate(klassen) if ZIEL_NAME.lower() in k.lower()]
    if not treffer:
        raise ValueError(f"Keine Klasse mit '{ZIEL_NAME}' gefunden")
    ziel = treffer[0]
print(f"\nErklärte Klasse: {klassen[ziel]}  (P = {wahrsch[ziel]:.3f})")

# ============================================================
# 4. Bild in N x N Felder teilen (Masken liegen auf der GPU)
# ============================================================
n_spieler = N * N
grenzen = np.linspace(0, 224, N + 1).astype(int)
feld_masken = []
for r in range(N):
    for c in range(N):
        m = torch.zeros(224, 224)
        m[grenzen[r]:grenzen[r + 1], grenzen[c]:grenzen[c + 1]] = 1
        feld_masken.append(m)
feld_masken = torch.stack(feld_masken).to(device)

# ============================================================
# 5. probly: Dropout-Modell (nach dem Verschieben von vgg auf die GPU bauen)
# ============================================================
mc  = dropout(vgg, p=0.2, predictor_type="logit_classifier")
rep = representer(mc, num_samples=NUM_SAMPLES)

def bewerte(koalitionen, batch=16):
    k = torch.tensor(koalitionen, dtype=torch.float32, device=device)
    ps, us = [], []
    for start in range(0, len(k), batch):
        teil = k[start:start + batch]
        maske = (teil[:, :, None, None] * feld_masken[None]).sum(1)
        bilder = x * maske[:, None]
        with torch.no_grad():
            sample = rep.represent(bilder)
            ps.append(sample.sample_mean().probabilities[:, ziel].detach().cpu().numpy())
            us.append(quantify(sample).total.detach().cpu().numpy())
    return np.concatenate(ps).astype(float), np.concatenate(us).astype(float)

cache = {}
def hole(koal):
    keys = [tuple(bool(b) for b in z) for z in koal]
    fehlend = [k for k in dict.fromkeys(keys) if k not in cache]
    if fehlend:
        p, u = bewerte(np.array(fehlend))
        for k, pv, uv in zip(fehlend, p, u):
            cache[k] = (pv, uv)
    return keys

def spiel_p(koal): return np.array([cache[k][0] for k in hole(koal)], dtype=float)
def spiel_u(koal): return np.array([cache[k][1] for k in hole(koal)], dtype=float)

# ============================================================
# 6. shapiq
# ============================================================
def shapley(spiel):
    if n_spieler <= 10:
        sv = shapiq.ExactComputer(game=spiel, n_players=n_spieler)(index="SV", order=1)
    else:
        sv = KernelSHAP(n=n_spieler, random_state=0).approximate(budget=BUDGET, game=spiel)
    werte = np.array([sv.dict_values.get((i,), 0.0) for i in range(n_spieler)])
    return werte, sv.baseline_value

werte_p, basis_p = shapley(spiel_p)
werte_u, basis_u = shapley(spiel_u)

voll = tuple([True] * n_spieler)
spiel_p(np.array([voll]))
P_voll, U_voll = cache[voll]

print("\n=== Beitrag zur WAHRSCHEINLICHKEIT ===")
print(np.round(werte_p.reshape(N, N), 3))
print(f"Basis {basis_p:.4f} + Summe {werte_p.sum():.4f} = {basis_p + werte_p.sum():.4f}   (ganzes Bild: {P_voll:.4f})")
print("\n=== Beitrag zur UNSICHERHEIT ===")
print(np.round(werte_u.reshape(N, N), 4))
print(f"Basis {basis_u:.5f} + Summe {werte_u.sum():.5f} = {basis_u + werte_u.sum():.5f}   (ganzes Bild: {U_voll:.5f})")

# ============================================================
# 7. Grafik
# ============================================================
karte_p, karte_u = werte_p.reshape(N, N), werte_u.reshape(N, N)
bild_224 = bild.resize((224, 224))
schrift = 8 if N > 3 else 10

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes[0, 0].imshow(bild_224); axes[0, 0].axis("off")
axes[0, 0].set_title(f"{klassen[ziel]}\nP = {P_voll:.2f}   Unsicherheit = {U_voll:.4f}")

for ax, karte, titel, fmt in [(axes[0, 1], karte_p, "shapiq: Beitrag zur Wahrscheinlichkeit", "{:.2f}"),
                              (axes[0, 2], karte_u, "shapiq: Beitrag zur Unsicherheit", "{:.3f}")]:
    g = np.abs(karte).max()
    im = ax.imshow(karte, cmap="RdBu_r", vmin=-g, vmax=g)
    for r in range(N):
        for c in range(N):
            ax.text(c, r, fmt.format(karte[r, c]), ha="center", va="center", fontsize=schrift)
    ax.set_title(titel); ax.axis("off"); fig.colorbar(im, ax=ax, fraction=0.046)

for ax, karte, titel in [(axes[1, 0], karte_p, "Überlagerung: Wahrscheinlichkeit"),
                         (axes[1, 1], karte_u, "Überlagerung: Unsicherheit")]:
    g = np.abs(karte).max()
    ax.imshow(bild_224)
    ax.imshow(karte, cmap="RdBu_r", vmin=-g, vmax=g, alpha=0.5, extent=(0, 224, 224, 0), interpolation="nearest")
    ax.set_title(titel); ax.axis("off")

ax = axes[1, 2]
ax.scatter(werte_p, werte_u, s=50)
for i in range(n_spieler):
    ax.annotate(str(i), (werte_p[i], werte_u[i]), textcoords="offset points", xytext=(4, 4), fontsize=8)
ax.axhline(0, color="gray", lw=0.8); ax.axvline(0, color="gray", lw=0.8)
ax.set_xlabel("Beitrag zur Wahrscheinlichkeit"); ax.set_ylabel("Beitrag zur Unsicherheit")
ax.set_title("Jedes Feld: wichtig für die Klasse? treibt es die Unsicherheit?")
plt.tight_layout(); plt.show()

print(f"\nWichtigstes Feld für die Klasse:           Feld {int(np.argmax(werte_p))}  ({werte_p.max():+.4f})")
print(f"Feld mit größtem Beitrag zur Unsicherheit: Feld {int(np.argmax(werte_u))}  ({werte_u.max():+.5f})")
print(f"Korrelation der beiden Beitragslisten:     {np.corrcoef(werte_p, werte_u)[0, 1]:+.3f}")

Rechnet auf: mps


FileNotFoundError: Bild nicht gefunden: andrew-s-ouo1hbizWwo-unsplash-scaled.jpg
Aktueller Ordner: /